# Prueba final H — Optimización controlada de `learning_rate` en DDPG-R2

**Objetivo de la entrega:** determinar si cambiar solamente la tasa de aprendizaje mejora la estabilidad estricta. **No se cambia la recompensa, la dinámica ni la definición de éxito.**

**Diseño:** tres semillas independientes 41/42/43, cada una con el mismo punto de partida R2-80K y dos tratamientos: LR=3e-4 (referencia H02) y LR=1e-4 (alternativa). A cada rama se le agregan 80K pasos de aprendizaje con nuevo replay buffer y calentamiento de 5K, igual que el Experimento E. La comparación es de **ajuste fino desde una política común**, no un ensayo desde cero de todo el entrenamiento. El entrenamiento original hasta 80K es compartido por ambas ramas de cada semilla.

**Preselección:** checkpoints 100K,120K,140K,160K validados en semillas 6000–6004. Selección de checkpoint por máxima estabilidad estricta media de validación, desempate por ocupación. **Prueba reservada:** 9100–9109 (no usar para elegir el modelo). Medir diferencias emparejadas por semilla *de entrenamiento*; n=3, evidencia exploratoria.

**Tiempo:** 6 continuaciones de 80K pasos (480K adicionales) y evaluaciones. Si Colab interrumpe, se conservan modelos guardados pero es importante completar las 3 réplicas de las 2 ramas antes de declarar ganador. No repetir el entrenamiento de 0-80K.

## 1. Instalar dependencias en Colab

In [ ]:
%pip -q install 'stable-baselines3>=2.3,<3' 'gymnasium>=0.29,<2' 'pandas>=2' 'matplotlib>=3'

## 1. Dependencias y configuración de referencia

In [ ]:
from pathlib import Path
import os, json, math, hashlib, time, random, zipfile, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym
import stable_baselines3 as sb3
from stable_baselines3 import DDPG
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.noise import NormalActionNoise
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.utils import set_random_seed
from google.colab import files

OUT=Path('resultados_H_LR_R2'); OUT.mkdir(parents=True,exist_ok=True)
CFG=dict(train_seeds=[41,42,43],memory_warmup=5000,continue_steps=80000,
         checkpoints=[100000,120000,140000,160000],val_seeds=[6000,6001,6002,6003,6004],
         test_seeds=list(range(9100,9110)),episode_steps=6000,dt=0.02,
         theta_bono_rad=.15,omega_bono_r1_rad_s=.50,x_bono_r2_m=.35,
         v_bono_r2_m_s=.25,bono_r2=3.0)
H02=dict(learning_rate=3e-4,gamma=.995,tau=.01,batch_size=128,sigma=.10,net_arch=[128,128])
LR_LIST=[('LR_3e-4',3e-4),('LR_1e-4',1e-4)]
print('SB3',sb3.__version__,'Gymnasium',gym.__version__)
print('Cambios experimentales: solo learning_rate; presupuesto adicional',6*80000,'pasos')


## 3. Entorno físico original — sin cambios

Esta celda incorpora el código del entorno usado en los experimentos B, C y D. El estado observado es BASE4. La dinámica y el código de reinicio se mantienen. La fuente se exportará en el paquete final.

In [ ]:
from pathlib import Path
Path('envs').mkdir(exist_ok=True)
Path('envs/__init__.py').write_text('',encoding='utf-8')
ORIGINAL_ENV = '"""\nEntorno Gymnasium: péndulo invertido tipo carro–poste con acción continua.\n\nEstado:\n    s_t = [x_t, x_dot_t, theta_t, theta_dot_t]\n\nAcción:\n    a_t = u(t), fuerza continua aplicada al carro.\n\nConvención angular:\n    theta = 0  -> péndulo vertical hacia arriba.\n    theta = pi -> péndulo hacia abajo.\n"""\n\nfrom __future__ import annotations\n\nimport math\nfrom typing import Optional\n\nimport gymnasium as gym\nfrom gymnasium import spaces\nimport numpy as np\n\n\nclass CartPoleContinuousEnv(gym.Env):\n    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 50}\n\n    def __init__(\n        self,\n        render_mode: Optional[str] = None,\n        max_episode_steps: int = 1000,\n        force_max: float = 10.0,\n        dt: float = 0.02,\n        x_threshold: float = 2.4,\n        noise_std: float = 0.0,\n        viscous_friction_cart: float = 0.0,\n        viscous_friction_pole: float = 0.0,\n        external_impulse: float = 0.0,\n        impulse_step: Optional[int] = None,\n        random_initial_state: bool = True,\n        start_near_downward: bool = True,\n    ):\n        super().__init__()\n\n        self.gravity = 9.81\n        self.masscart = 1.0\n        self.masspole = 0.1\n        self.total_mass = self.masscart + self.masspole\n        self.length = 0.5\n        self.polemass_length = self.masspole * self.length\n\n        self.force_max = float(force_max)\n        self.dt = float(dt)\n        self.x_threshold = float(x_threshold)\n        self.max_episode_steps = int(max_episode_steps)\n\n        self.noise_std = float(noise_std)\n        self.viscous_friction_cart = float(viscous_friction_cart)\n        self.viscous_friction_pole = float(viscous_friction_pole)\n        self.external_impulse = float(external_impulse)\n        self.impulse_step = impulse_step\n\n        self.random_initial_state = random_initial_state\n        self.start_near_downward = start_near_downward\n\n        self.render_mode = render_mode\n        self.screen = None\n        self.clock = None\n        self.screen_width = 800\n        self.screen_height = 500\n\n        self.action_space = spaces.Box(\n            low=np.array([-self.force_max], dtype=np.float32),\n            high=np.array([self.force_max], dtype=np.float32),\n            dtype=np.float32,\n        )\n\n        high = np.array(\n            [\n                self.x_threshold * 2.0,\n                np.finfo(np.float32).max,\n                math.pi,\n                np.finfo(np.float32).max,\n            ],\n            dtype=np.float32,\n        )\n        self.observation_space = spaces.Box(-high, high, dtype=np.float32)\n\n        self.state: Optional[np.ndarray] = None\n        self.steps = 0\n        self.last_u = 0.0\n        self.cumulative_control_effort = 0.0\n        self.cumulative_reward = 0.0\n\n    @staticmethod\n    def angle_normalize(theta: float) -> float:\n        return ((theta + math.pi) % (2.0 * math.pi)) - math.pi\n\n    def _get_obs(self) -> np.ndarray:\n        assert self.state is not None\n        obs = self.state.copy()\n\n        if self.noise_std > 0:\n            noise = self.np_random.normal(0.0, self.noise_std, size=obs.shape)\n            obs = obs + noise.astype(np.float32)\n\n        obs[2] = self.angle_normalize(float(obs[2]))\n        return obs.astype(np.float32)\n\n    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):\n        super().reset(seed=seed)\n        self.steps = 0\n        self.last_u = 0.0\n        self.cumulative_control_effort = 0.0\n        self.cumulative_reward = 0.0\n\n        if self.random_initial_state:\n            if self.start_near_downward:\n                x = self.np_random.uniform(low=-0.05, high=0.05)\n                x_dot = self.np_random.uniform(low=-0.05, high=0.05)\n                theta = math.pi + self.np_random.uniform(low=-0.10, high=0.10)\n                theta_dot = self.np_random.uniform(low=-0.05, high=0.05)\n            else:\n                x = self.np_random.uniform(low=-0.5, high=0.5)\n                x_dot = self.np_random.uniform(low=-0.5, high=0.5)\n                theta = self.np_random.uniform(low=-math.pi, high=math.pi)\n                theta_dot = self.np_random.uniform(low=-1.0, high=1.0)\n        else:\n            x, x_dot, theta, theta_dot = 0.0, 0.0, math.pi, 0.0\n\n        theta = self.angle_normalize(theta)\n        self.state = np.array([x, x_dot, theta, theta_dot], dtype=np.float32)\n\n        info = {\n            "cumulative_control_effort": self.cumulative_control_effort,\n            "cumulative_reward": self.cumulative_reward,\n        }\n        return self._get_obs(), info\n\n    def step(self, action):\n        assert self.state is not None\n\n        x, x_dot, theta, theta_dot = map(float, self.state)\n        force = float(np.clip(action[0], -self.force_max, self.force_max))\n\n        if self.impulse_step is not None and self.steps == self.impulse_step:\n            force += self.external_impulse\n\n        force -= self.viscous_friction_cart * x_dot\n\n        costheta = math.cos(theta)\n        sintheta = math.sin(theta)\n\n        temp = (\n            force + self.polemass_length * theta_dot**2 * sintheta\n        ) / self.total_mass\n\n        theta_acc = (\n            self.gravity * sintheta\n            - costheta * temp\n            - self.viscous_friction_pole * theta_dot / max(self.polemass_length, 1e-8)\n        ) / (\n            self.length * (4.0 / 3.0 - self.masspole * costheta**2 / self.total_mass)\n        )\n\n        x_acc = temp - self.polemass_length * theta_acc * costheta / self.total_mass\n\n        x_dot = x_dot + self.dt * x_acc\n        x = x + self.dt * x_dot\n        theta_dot = theta_dot + self.dt * theta_acc\n        theta = theta + self.dt * theta_dot\n        theta = self.angle_normalize(theta)\n\n        self.state = np.array([x, x_dot, theta, theta_dot], dtype=np.float32)\n        self.steps += 1\n\n        reward = self._reward(x, x_dot, theta, theta_dot, force)\n\n        terminated = bool(abs(x) > self.x_threshold)\n        truncated = bool(self.steps >= self.max_episode_steps)\n\n        self.last_u = force\n        self.cumulative_control_effort += force**2 * self.dt\n        self.cumulative_reward += reward\n\n        info = {\n            "x": x,\n            "x_dot": x_dot,\n            "theta": theta,\n            "theta_dot": theta_dot,\n            "u": force,\n            "cumulative_control_effort": self.cumulative_control_effort,\n            "cumulative_reward": self.cumulative_reward,\n            "is_upright": abs(theta) < 0.15,\n        }\n\n        if self.render_mode == "human":\n            self.render()\n\n        return self._get_obs(), float(reward), terminated, truncated, info\n\n    def _reward(self, x: float, x_dot: float, theta: float, theta_dot: float, u: float) -> float:\n        """\n        Recompensa mejorada para swing-up y estabilización.\n        """\n\n        reward = 1.0 + math.cos(theta)\n\n        reward += 0.05\n\n        reward -= 0.20 * x**2\n        reward -= 0.01 * x_dot**2\n        reward -= 0.02 * theta_dot**2\n        reward -= 0.001 * u**2\n\n        if abs(theta) < 0.15:\n            reward += 3.0\n\n        if abs(x) > 0.80 * self.x_threshold:\n            reward -= 5.0 * (abs(x) - 0.80 * self.x_threshold)\n\n        if abs(x) > self.x_threshold:\n            reward -= 100.0\n\n        return reward\n\n    def render(self):\n        try:\n            import pygame\n            from pygame import gfxdraw\n        except ImportError as exc:\n            raise ImportError("Para renderizar instala pygame: pip install pygame") from exc\n\n        if self.screen is None:\n            pygame.init()\n            if self.render_mode == "human":\n                self.screen = pygame.display.set_mode((self.screen_width, self.screen_height))\n            else:\n                self.screen = pygame.Surface((self.screen_width, self.screen_height))\n            self.clock = pygame.time.Clock()\n\n        assert self.state is not None\n        x, _, theta, _ = self.state\n\n        world_width = self.x_threshold * 2\n        scale = self.screen_width / world_width\n        cart_y = int(self.screen_height * 0.65)\n        cart_w = 60\n        cart_h = 30\n        pole_len = scale * (2 * self.length)\n\n        cart_x = int(self.screen_width / 2 + x * scale)\n\n        self.screen.fill((255, 255, 255))\n        import pygame\n\n        pygame.draw.line(\n            self.screen, (0, 0, 0),\n            (0, cart_y + cart_h // 2),\n            (self.screen_width, cart_y + cart_h // 2),\n            2\n        )\n\n        cart_rect = pygame.Rect(cart_x - cart_w // 2, cart_y - cart_h // 2, cart_w, cart_h)\n        pygame.draw.rect(self.screen, (30, 144, 255), cart_rect)\n        pygame.draw.rect(self.screen, (0, 0, 0), cart_rect, 2)\n\n        pivot = (cart_x, cart_y - cart_h // 2)\n        pole_x = pivot[0] + pole_len * math.sin(theta)\n        pole_y = pivot[1] - pole_len * math.cos(theta)\n\n        pygame.draw.line(self.screen, (220, 80, 20), pivot, (pole_x, pole_y), 6)\n        gfxdraw.filled_circle(self.screen, int(pole_x), int(pole_y), 10, (220, 80, 20))\n        gfxdraw.aacircle(self.screen, int(pole_x), int(pole_y), 10, (0, 0, 0))\n\n        font = pygame.font.SysFont("Arial", 18)\n        txt = font.render(f"theta={theta:.2f} rad | u={self.last_u:.2f} N", True, (0, 0, 0))\n        self.screen.blit(txt, (20, 20))\n\n        if self.render_mode == "human":\n            pygame.event.pump()\n            pygame.display.flip()\n            self.clock.tick(self.metadata["render_fps"])\n        else:\n            return np.transpose(\n                np.array(pygame.surfarray.pixels3d(self.screen)), axes=(1, 0, 2)\n            )\n\n    def close(self):\n        if self.screen is not None:\n            import pygame\n            pygame.display.quit()\n            pygame.quit()\n            self.screen = None\n            self.clock = None\n'
Path('envs/cartpole_continuous_env.py').write_text(ORIGINAL_ENV,encoding='utf-8')
print('Entorno original preparado; SHA-256:',hashlib.sha256(ORIGINAL_ENV.encode('utf-8')).hexdigest())
from envs.cartpole_continuous_env import CartPoleContinuousEnv

## 4. Recompensa R2 (exactamente la misma del Experimento C y D)

In [ ]:
class CartPoleRewardR1(CartPoleContinuousEnv):
    """R1: bono +3 solo cerca de la vertical y con |omega|<0.50."""
    def _reward(self,x,x_dot,theta,theta_dot,u):
        r=super()._reward(x,x_dot,theta,theta_dot,u)
        if abs(theta)<CFG['theta_bono_rad'] and abs(theta_dot)>=CFG['omega_bono_r1_rad_s']:
            r-=3.0
        return r

class CartPoleRewardR2(CartPoleRewardR1):
    """R2: R1 + bono de equilibrio local de cuatro variables."""
    def _reward(self,x,x_dot,theta,theta_dot,u):
        r=super()._reward(x,x_dot,theta,theta_dot,u)
        cuatro=(abs(x)<CFG['x_bono_r2_m'] and abs(x_dot)<CFG['v_bono_r2_m_s']
                and abs(theta)<CFG['theta_bono_rad'] and abs(theta_dot)<CFG['omega_bono_r1_rad_s'])
        return r+(CFG['bono_r2'] if cuatro else 0.0)

def make_env(variante,steps=6000):
    cls={'R0':CartPoleContinuousEnv,'R1':CartPoleRewardR1,'R2':CartPoleRewardR2}[variante]
    return cls(max_episode_steps=steps,force_max=10.0,dt=0.02,
      x_threshold=2.4,noise_std=0.0,viscous_friction_cart=0.0,
      viscous_friction_pole=0.0,random_initial_state=True,start_near_downward=True)

# Pruebas del bono R2: SOLO depende de las cuatro variables de estado
r1=make_env('R1');r2=make_env('R2')
casos=[(0,0,0,0,3.0),(.36,0,0,0,0.0),(0,.26,0,0,0.0),
       (0,0,.16,0,0.0),(0,0,0,.51,0.0),(.1,.1,.1,.3,3.0)]
for x,v,th,w,esperado in casos:
    dif=r2._reward(x,v,th,w,0)-r1._reward(x,v,th,w,0)
    assert np.isclose(dif,esperado), (x,v,th,w,dif,esperado)
for semilla in [41,7000]:
    a,_=r1.reset(seed=semilla);b,_=r2.reset(seed=semilla)
    assert np.array_equal(a,b)
    for fuerza in [3.,-2.,1.]:
        oa,rew1,term1,trunc1,_=r1.step(np.array([fuerza],dtype=np.float32))
        ob,rew2,term2,trunc2,_=r2.step(np.array([fuerza],dtype=np.float32))
        assert np.array_equal(oa,ob) and term1==term2 and trunc1==trunc2
        if term1 or trunc1: break
r1.close();r2.close()
print('OK: R1 y R2 difieren exclusivamente por el bono local de cuatro estados.')
print('R2 = R1 + 3 si |x|<0.35, |v|<0.25, |theta|<0.15, |omega|<0.50')


## 5. Métricas y protocolo de evaluación: políticas congeladas sobre R0

**Criterio estricto:** |x| ≤ 0,20 m, |v| ≤ 0,10 m/s, |θ| ≤ 0,05 rad, |ω| ≤ 0,10 rad/s. El *RMSE angular* es la raíz del promedio de θ², sin términos ajenos. Se cuantifica también la captura sostenida ≥1 s.

In [ ]:
def evaluar(modelo,semilla,steps=6000,guardar_traza=False):
    # Se usa siempre el entorno entorno original R0 durante la evaluación: la métrica NO depende de la recompensa entrenada.
    env=make_env('R0',steps)
    obs,_=env.reset(seed=int(semilla))
    largo=actual=estable=pasos_cerca=cruces=0
    capturas_1s=0; previo_capturado=False; fraccion_local=0
    cumplen={"x":0,"v":0,"theta":0,"omega":0}
    prev_theta=float(env.state[2])
    rmse_acum=cos_acum=esfuerzo=vel_cruce=0.0
    velocidades_cruce=[]; traza=[];termina_riel=False
    for k in range(steps):
        accion,_=modelo.predict(obs,deterministic=True)
        u=float(np.asarray(accion).reshape(-1)[0])
        obs,_,term,trunc,info=env.step(np.array([u],dtype=np.float32))
        x,v,th,w=map(float,env.unwrapped.state)
        dentro=abs(x)<=.20 and abs(v)<=.10 and abs(th)<=.05 and abs(w)<=.10
        cumplen['x']+=int(abs(x)<=.20)
        cumplen['v']+=int(abs(v)<=.10)
        cumplen['theta']+=int(abs(th)<=.05)
        cumplen['omega']+=int(abs(w)<=.10)
        local=(abs(x)<CFG['x_bono_r2_m'] and abs(v)<CFG['v_bono_r2_m_s'] and abs(th)<CFG['theta_bono_rad'] and abs(w)<CFG['omega_bono_r1_rad_s'])
        fraccion_local+=int(local)
        actual=actual+1 if dentro else 0
        if actual==50: capturas_1s+=1
        largo=max(largo,actual); estable+=int(dentro)
        pasos_cerca+=int(abs(th)<.15)
        rmse_acum+=th*th;cos_acum+=math.cos(th);esfuerzo+=u*u*CFG['dt']
        # Cruce próximo a vertical, detección aproximada por cambio de signo y no por wrap de +/-pi
        if (prev_theta*th<=0 and abs(prev_theta)<.6 and abs(th)<.6):
            cruces+=1; velocidades_cruce.append(abs(w))
        prev_theta=th
        if guardar_traza:
            traza.append(dict(t=(k+1)*CFG['dt'],x=x,v=v,theta=th,omega=w,u=u,
                               estricta=int(dentro),equilibrio_local=int(local),cerca_vertical=int(abs(th)<.15)))
        if term or trunc:
            termina_riel=bool(term); break
    n=k+1;env.close()
    resultado=dict(seed=int(semilla),estabilidad_max_s=largo*CFG['dt'],
      ocupacion_pct=100*estable/n,supervivencia_s=n*CFG['dt'],
      rmse_theta_rad=math.sqrt(rmse_acum/n),cos_theta_medio=cos_acum/n,
      esfuerzo_N2s=esfuerzo,esfuerzo_medio_N2=esfuerzo/(n*CFG['dt']),
      fraccion_cerca_vertical=pasos_cerca/n,fin_por_riel=termina_riel,
      fraccion_equilibrio_local_pct=100*fraccion_local/n, capturas_1s=int(capturas_1s),
      **{f'cumple_{nombre}_pct':100*valor/n for nombre,valor in cumplen.items()},
      cruces_vertical=cruces,
      velocidad_angular_cruce_mediana_rad_s=(float(np.median(velocidades_cruce)) if velocidades_cruce else np.nan))
    return (resultado,pd.DataFrame(traza)) if guardar_traza else resultado

def score(row):
    # Score diagnóstico predefinido; no reemplaza la métrica primaria de estabilidad
    return (row['estabilidad_max_s']/(CFG['episode_steps']*CFG['dt'])
            +0.5*row['ocupacion_pct']/100
            +0.25*(row['cos_theta_medio']+1)/2
            -0.2*float(row['fin_por_riel']))
print('Evaluación independiente preparada: ambos controladores se prueban sobre R0 con pesos congelados.')

## 2. Cargar modelos R2-80K originales (Experimento E)

Sube **`experimento_E_R2_160K_multisemilla_resultados_modelos.zip`**. Se extraerán **solo** los checkpoints a 80K. Verifica que los SHA256 y versiones coincidan con los archivos originales antes de aceptar comparaciones.

In [ ]:
print('Selecciona el ZIP del Experimento E:')
uploaded=files.upload()
assert len(uploaded)==1,'Sube exactamente un ZIP'
source=Path(next(iter(uploaded)))
INPUT=OUT/'base_80K';INPUT.mkdir(exist_ok=True)
expected={s:f'resultados_experimento_E_R2_multisemilla/seed_{s}/modelo_80000.zip' for s in CFG['train_seeds']}
with zipfile.ZipFile(source) as z:
    for s,name in expected.items():
        assert name in z.namelist(),f'Falta {name}'
        (INPUT/f'modelo_80000_seed_{s}.zip').write_bytes(z.read(name))
        mo=DDPG.load(str(INPUT/f'modelo_80000_seed_{s}.zip'),device='cpu')
        assert mo.num_timesteps==80000
        assert mo.observation_space.shape==(4,)
        assert np.isclose(mo.gamma,H02['gamma']) and np.isclose(mo.tau,H02['tau'])
        assert mo.batch_size==H02['batch_size']
        print(f'Semilla {s}: contador={mo.num_timesteps}, gamma={mo.gamma}, tau={mo.tau}, batch={mo.batch_size}')
print('Modelos preparados')


## 3. Entrenamiento pareado: una sola tasa de aprendizaje cambia

Cada continuación parte **del mismo archivo R2-80K** para la respectiva semilla. El optimizador conserva el estado del checkpoint y se actualiza únicamente su tasa de aprendizaje. Igualamos la semilla de la continuación dentro del par. Se recalienta una memoria nueva hasta 85K globales para evitar actualizaciones sin suficientes experiencias. Usamos `reset_num_timesteps=False`. Se guardan checkpoints y un registro por rama.

In [ ]:
class GuardarPasos(BaseCallback):
    def __init__(self,ruta,pasos):
        super().__init__();self.ruta=Path(ruta);self.pasos=set(pasos)
    def _on_step(self):
        t=int(self.model.num_timesteps)
        if t in self.pasos:
            self.model.save(str(self.ruta/f'modelo_{t}.zip'))
            print(f'    checkpoint {t:,}',flush=True)
        return True

def ajustar_lr(model,lr):
    # Importante: modificar schedule y optimizadores, no solamente model.learning_rate
    model.learning_rate=float(lr)
    model.lr_schedule=lambda _: float(lr)
    for opt in [model.actor.optimizer,model.critic.optimizer]:
        for group in opt.param_groups:group['lr']=float(lr)
    assert all(abs(g['lr']-lr)<1e-12 for opt in
               [model.actor.optimizer,model.critic.optimizer] for g in opt.param_groups)

def ejecutar_rama(seed,etiqueta,lr):
    ruta=OUT/f'seed_{seed}'/etiqueta;ruta.mkdir(parents=True,exist_ok=True)
    if (ruta/'modelo_160000.zip').exists() and (ruta/'registro.json').exists():
        reg=json.loads((ruta/'registro.json').read_text())
        if reg.get('completed'):
            print('Reutilizando rama',seed,etiqueta);return
    print('\nENTRENANDO',seed,etiqueta,lr,flush=True)
    t0=time.perf_counter()
    set_random_seed(10000+int(seed))  # exactamente el mismo número en ambas ramas
    env=Monitor(make_env('R2',CFG['episode_steps']),filename=str(ruta/'monitor'))
    env.reset(seed=int(seed))
    model=DDPG.load(str(INPUT/f'modelo_80000_seed_{seed}.zip'),device='cpu')
    assert model.num_timesteps==80000
    assert model.replay_buffer is None or model.replay_buffer.size()==0
    model.set_env(env,force_reset=True)
    ajustar_lr(model,lr)
    model.action_noise=NormalActionNoise(mean=np.zeros(1),sigma=np.ones(1)*H02['sigma'])
    model.learning_starts=85000
    try:
        model.learn(total_timesteps=CFG['continue_steps'],reset_num_timesteps=False,
                    callback=GuardarPasos(ruta,CFG['checkpoints']),progress_bar=False)
        assert model.num_timesteps==160000
        assert model.replay_buffer.size()>0
    finally: env.close()
    reg=dict(seed=seed,etiqueta=etiqueta,learning_rate=lr,completed=True,
             steps_start=80000,steps_end=160000,warmup_start_updates=85000,
             minutes=round((time.perf_counter()-t0)/60,3))
    (ruta/'registro.json').write_text(json.dumps(reg,indent=2))
    print('OK:',reg,flush=True)

for seed in CFG['train_seeds']:
    for etiqueta,lr in LR_LIST:
        ejecutar_rama(seed,etiqueta,lr)
print('Se completaron las seis continuaciones')


## 4. Validación de checkpoints y selección ANTES de prueba reservada

In [ ]:
rows=[]
for seed in CFG['train_seeds']:
    for etiqueta,_ in LR_LIST:
        for paso in CFG['checkpoints']:
            p=OUT/f'seed_{seed}'/etiqueta/f'modelo_{paso}.zip'
            assert p.exists(),f'Modelo faltante: {p}'
            model=DDPG.load(str(p),device='cpu')
            for es in CFG['val_seeds']:
                res=evaluar(model,es,CFG['episode_steps'],guardar_traza=False)
                rows.append(dict(train_seed=seed,tratamiento=etiqueta,checkpoint=paso,
                                 eval_seed=es,**{k:v for k,v in res.items() if k!='seed'}))
valid=pd.DataFrame(rows)
valid.to_csv(OUT/'validacion_checkpoints.csv',index=False)
summary=valid.groupby(['train_seed','tratamiento','checkpoint'],as_index=False)[
    ['estabilidad_max_s','ocupacion_pct','rmse_theta_rad','cumple_x_pct']].mean()
summary.to_csv(OUT/'resumen_validacion.csv',index=False)
selected=(summary.sort_values(['train_seed','tratamiento','estabilidad_max_s','ocupacion_pct','checkpoint'],
                               ascending=[True,True,False,False,True])
                 .drop_duplicates(['train_seed','tratamiento']))
selected.to_csv(OUT/'checkpoints_seleccionados_SOLO_VALIDACION.csv',index=False)
print(selected.to_string(index=False))


## 5. Prueba final reservada 9100–9109

No modificar la selección de checkpoints después de ver esta sección. La unidad de comparación es la **semilla de entrenamiento**, no los 30 episodios como si fueran 30 entrenamientos independientes.

In [ ]:
rows=[]
for row in selected.itertuples(index=False):
    path=OUT/f'seed_{row.train_seed}'/row.tratamiento/f'modelo_{row.checkpoint}.zip'
    model=DDPG.load(str(path),device='cpu')
    for es in CFG['test_seeds']:
        guardar=(es==CFG['test_seeds'][0])
        salida=evaluar(model,es,CFG['episode_steps'],guardar_traza=guardar)
        if guardar:
            res,traza=salida
            traza.to_csv(OUT/f'trayectoria_{row.tratamiento}_train_{row.train_seed}_eval_{es}.csv',index=False)
        else:res=salida
        rows.append(dict(train_seed=int(row.train_seed),tratamiento=row.tratamiento,
                         checkpoint=int(row.checkpoint),eval_seed=es,
                         **{k:v for k,v in res.items() if k!='seed'}))
test=pd.DataFrame(rows);test.to_csv(OUT/'prueba_reservada_9100_9109.csv',index=False)
metrics=['estabilidad_max_s','ocupacion_pct','rmse_theta_rad','cumple_x_pct','supervivencia_s','esfuerzo_medio_N2']
by_policy=test.groupby(['train_seed','tratamiento'],as_index=False)[metrics].mean()
by_policy.to_csv(OUT/'resumen_por_politica.csv',index=False)
a=by_policy.query('tratamiento=="LR_3e-4"').set_index('train_seed')
b=by_policy.query('tratamiento=="LR_1e-4"').set_index('train_seed')
delta=(b[metrics]-a[metrics]).reset_index()
delta.to_csv(OUT/'diferencias_LR_baja_menos_referencia.csv',index=False)
print('RESULTADOS por entrenamiento:')
print(by_policy.round(3).to_string(index=False))
print('\nDIFERENCIAS LR 1e-4 MENOS 3e-4 (por semilla):')
print(delta.round(3).to_string(index=False))
print('\nMedia de diferencia estricta:',delta['estabilidad_max_s'].mean().round(3),'s')
print('Desviación estándar:',delta['estabilidad_max_s'].std(ddof=1).round(3),'s')
print('Mejoras en:',int((delta.estabilidad_max_s>0).sum()),'de 3 semillas')


## 6. Gráficas, trazabilidad y ZIP final

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4))
for i,metrica in enumerate(['estabilidad_max_s','ocupacion_pct']):
    for seed in CFG['train_seeds']:
        tmp=by_policy.query('train_seed==@seed').set_index('tratamiento')
        axs[i].plot(['LR_3e-4','LR_1e-4'],tmp.loc[['LR_3e-4','LR_1e-4'],metrica].to_numpy(),
                    marker='o',label=f'Semilla {seed}')
    axs[i].set_title('Estabilidad estricta' if i==0 else 'Ocupación estricta')
    axs[i].set_ylabel('segundos' if i==0 else '%');axs[i].grid(alpha=.3)
axs[0].legend();fig.tight_layout();fig.savefig(OUT/'comparacion_LR.png',dpi=150);plt.show()
manifest=dict(experimento='H — learning_rate controlado, R2',
    diseño='3 semillas, 2 tasas, warm-start pareado desde R2-80K; solo cambia LR',
    configuracion=CFG,hiperparametros_constantes=H02,tratamientos=LR_LIST,
    unidad_independiente='semilla de entrenamiento, n=3',
    conjunto_seleccion='validacion 6000-6004',conjunto_reservado='9100-9109',
    advertencia='Evidencia exploratoria; ramas comparten politica inicial, no son entrenamientos desde cero independientes')
(OUT/'manifiesto.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
archive=Path('experimento_H_LR_DDPG_R2_resultados.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=5) as z:
    for f in OUT.rglob('*'):
        if f.is_file(): z.write(f,f.as_posix())
print('ZIP generado:',archive,archive.stat().st_size/1e6,'MB')
files.download(str(archive))


## 7. Reglas para interpretar la prueba en el informe

- Si 1e-4 mejora estabilidad estricta media en **2 o 3 semillas**, documenta el efecto observado; *n=3* no permite demostrar superioridad universal.
- Si mejora solo la semilla 41, informa **resultado sensible a semilla**; no lo vendas como optimización reproducible.
- Si reduce RMSE angular pero no mejora las cuatro restricciones simultáneas, **no** digas que mejora estabilización integral.
- La validación selecciona un checkpoint por rama; el test reservado no se usa para recalibrar.
- Los 80K iniciales son compartidos; este estudio estima el efecto de `learning_rate` en **ajuste fino**, no de entrenar todo desde cero.
- Reporta el tiempo real de Colab y las versiones de software en el manifiesto; guarda el ZIP y no pierdas el notebook.
